### 03_embeddings
Extracts face data and creates embeddings
* Uses scripts/embeddings.py module

Input: 
* `data_processed/<dataset>/manifests/manifest_enroll.csv`
* `data_processed/<dataset>/manifests/manifest_probe.csv`

Output:
* Embedding shards are created and saved as `.npy` and `.csv` files:
    * `data_processed/<dataset>/embeddings/enroll`
    * `data_processed/<dataset>/embeddings/probe`
    * `data_processed/<dataset>/embeddings/train`
    * `data_processed/<dataset>/embeddings/val`



In [2]:
# ----- Imports and config -----
import sys, importlib
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
# Run embedding extraction + merge for train/val and enroll/probe
# - Purpose: extract face embeddings using facenet-pytorch InceptionResnetV1 model
# - Uses scripts/embeddings.py module
# - Embeddings are stored as sharded files and then merged for easier downstream use
# - Merged files can be used for training/evaluation later
# - Outputs are stored under ../data_processed/vggface2/embeddings/

from pathlib import Path
import sys, importlib

# Paths & params
OUT_TRAIN  = "../data_processed/vggface2/embeddings/train"
OUT_VAL    = "../data_processed/vggface2/embeddings/val"
OUT_ENROLL = "../data_processed/vggface2/embeddings/enroll"
OUT_PROBE  = "../data_processed/vggface2/embeddings/probe"

MAN_TRAIN  = "../data_processed/vggface2/manifests/manifest_train.csv"
MAN_VAL    = "../data_processed/vggface2/manifests/manifest_val.csv"
MAN_ENROLL = "../data_processed/vggface2/manifests/manifest_enroll.csv"
MAN_PROBE  = "../data_processed/vggface2/manifests/manifest_probe.csv"

BATCH_SIZE = 128
SHARD_SIZE = 2000
SAMPLE_LIMIT = None  # set to int for quick tests

# Ensure notebook can import the `scripts/embeddings.py` module
scripts_dir = Path.cwd() / "scripts"
if str(scripts_dir) not in sys.path:
    sys.path.append(str(scripts_dir))

import embeddings as embeddings_module
importlib.reload(embeddings_module)

# Helper to run embedding + merge + status 
def run_and_merge(manifest_path, out_dir, batch_size=BATCH_SIZE, shard_size=SHARD_SIZE, sample_limit=SAMPLE_LIMIT):
    print(f"=== Processing manifest: {manifest_path} -> {out_dir} ===")
    embeddings_module.embed_sharded(manifest_path, out_dir, batch_size=batch_size, shard_size=shard_size, sample_limit=sample_limit)
    embeddings_module.merge_shards(out_dir)
    embeddings_module.status(out_dir)
    print("")

# Run for train/val 
if Path(MAN_TRAIN).exists():
    run_and_merge(MAN_TRAIN, OUT_TRAIN)
else:
    print("Skipping train: manifest not found:", MAN_TRAIN)

if Path(MAN_VAL).exists():
    run_and_merge(MAN_VAL, OUT_VAL)
else:
    print("Skipping val: manifest not found:", MAN_VAL)

# Run for enroll/probe (new closed-set split)
if Path(MAN_ENROLL).exists():
    run_and_merge(MAN_ENROLL, OUT_ENROLL)
else:
    print("Skipping enroll: manifest not found:", MAN_ENROLL)

if Path(MAN_PROBE).exists():
    run_and_merge(MAN_PROBE, OUT_PROBE)
else:
    print("Skipping probe: manifest not found:", MAN_PROBE)

print("All done. Check outputs under ../data_processed/vggface2/embeddings/")